In [5]:
! pip install -Uq openai tiktoken tenacity

In [1]:
import mlflow
import openai
import os
from dotenv import load_dotenv
from config import MODEL_NAME, PROMPT_NAME, PROMPT_VERSION, REASONING
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:8080/"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "cord-v2-gpt5-medium"

load_dotenv()

True

In [2]:
system_prompt = """You are a Vision Language Model designed to extract structured data from invoice receipts.
Task:
Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

Requirements:
1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
2. Preserve exact formatting for all the extracted values.  
3. Do not output fields that lack data—omit empty keys.  
4. Do not add any information not present in the invoice.
5. In case of prices and currencies, ensure to maintain the original format without any modifications.

Schema:
{schema}

Output:
Return valid, minimal JSON matching this schema - no extraneous keys or null values.
"""

print(system_prompt)

You are a Vision Language Model designed to extract structured data from invoice receipts.
Task:
Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

Requirements:
1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
2. Preserve exact formatting for all the extracted values.  
3. Do not output fields that lack data—omit empty keys.  
4. Do not add any information not present in the invoice.
5. In case of prices and currencies, ensure to maintain the original format without any modifications.

Schema:
{schema}

Output:
Return valid, minimal JSON matching this schema - no extraneous keys or null values.



In [3]:
prompt = mlflow.register_prompt(
    name = "invoice-extraction-gpt5-prompt",
    template = system_prompt
)

prompt

/tmp/ipykernel_1227/2666037749.py:1: FutureWarning: The `mlflow.register_prompt` API is moved to the `mlflow.genai` namespace. Please use `mlflow.genai.register_prompt` instead. The original API will be removed in the future release.
  prompt = mlflow.register_prompt(
2025/08/16 06:27:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: invoice-extraction-gpt5-prompt, version 1


PromptVersion(name=invoice-extraction-gpt5-prompt, version=1, template=You are a Vision Language Mode...)

In [4]:
mlflow.load_prompt("prompts:/invoice-extraction-gpt5-prompt/1").template

/tmp/ipykernel_1227/3726660216.py:1: FutureWarning: The `mlflow.load_prompt` API is moved to the `mlflow.genai` namespace. Please use `mlflow.genai.load_prompt` instead. The original API will be removed in the future release.
  mlflow.load_prompt("prompts:/invoice-extraction-gpt5-prompt/1").template


'You are a Vision Language Model designed to extract structured data from invoice receipts.\nTask:\nConvert the invoice receipt into a well-formed JSON object strictly following the schema provided.\n\nRequirements:\n1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  \n2. Preserve exact formatting for all the extracted values.  \n3. Do not output fields that lack data—omit empty keys.  \n4. Do not add any information not present in the invoice.\n5. In case of prices and currencies, ensure to maintain the original format without any modifications.\n\nSchema:\n{schema}\n\nOutput:\nReturn valid, minimal JSON matching this schema - no extraneous keys or null values.\n'

In [2]:
def log_invoice_extraction_model():
    system_prompt = mlflow.load_prompt(f"prompts:/{PROMPT_NAME}/{PROMPT_VERSION}").template
    print("System Prompt: ", system_prompt)
    with mlflow.start_run():
        model_info = mlflow.openai.log_model(
            model=MODEL_NAME,
            reasoning={
                "effort": REASONING,
            },
            task=openai.chat.completions,
            name="model",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": system_prompt},
                        {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,{image_base64}"}},
                    ],
                }
            ],
        )

    return model_info.model_uri

In [3]:
log_invoice_extraction_model()

/tmp/ipykernel_1844/652882919.py:2: FutureWarning: The `mlflow.load_prompt` API is moved to the `mlflow.genai` namespace. Please use `mlflow.genai.load_prompt` instead. The original API will be removed in the future release.
  system_prompt = mlflow.load_prompt(f"prompts:/{PROMPT_NAME}/{PROMPT_VERSION}").template


System Prompt:  You are a Vision Language Model designed to extract structured data from invoice receipts.
Task:
Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

Requirements:
1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
2. Preserve exact formatting for all the extracted values.  
3. Do not output fields that lack data—omit empty keys.  
4. Do not add any information not present in the invoice.
5. In case of prices and currencies, ensure to maintain the original format without any modifications.

Schema:
{schema}

Output:
Return valid, minimal JSON matching this schema - no extraneous keys or null values.

🏃 View run vaunted-ox-832 at: http://localhost:8080/#/experiments/481396941930403223/runs/6862575f4dfa4b0dbee9650707a8da60
🧪 View experiment at: http://localhost:8080/#/experiments/481396941930403223


'models:/m-1b0300a8d4684d87aadfd2ededb0c29a'